In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv


In [2]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

!pip install sentence-transformers bert-score --quiet

import pandas as pd, numpy as np, torch, ast, json, gc
from sentence_transformers import SentenceTransformer, InputExample, CrossEncoder
from sentence_transformers.losses import MultipleNegativesRankingLoss
from torch.utils.data import DataLoader
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from bert_score import score as bert_score_compute
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

print(f'GPU disponibil: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Memorie GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 81.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26

/tmp/ipykernel_23/1522885238.py:8: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss


In [3]:
INPUT_PATH  = '/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv'
OUTPUT_PATH = '/kaggle/working/'

df = pd.read_csv(INPUT_PATH)
print(f'Filme incarcate: {len(df):,}')
print(f'Coloane: {df.columns.tolist()}')

df['overview']         = df['overview'].fillna('').astype(str).str.strip()
df['tagline']          = df['tagline'].fillna('').astype(str).str.strip()
df['review_summary']   = df['review_summary'].fillna('').astype(str).str.strip().replace('nan', '')
df['overview_summary'] = df['overview_summary'].fillna('').astype(str).str.strip().replace('nan', '')
df['genres']           = df['genres'].fillna('').astype(str).str.strip()
df['keywords']         = df['keywords'].fillna('').astype(str).str.strip()
df['cast']             = df['cast'].fillna('').astype(str).str.strip()
df['director']         = df['director'].fillna('').astype(str).str.strip()

print()
for col in ['overview', 'overview_summary', 'review_summary', 'tagline', 'keywords']:
    n   = (df[col] != '').sum()
    avg = df[col].str.split().str.len().mean()
    print(f'  {col:<22} acoperire: {n:>6,} ({n/len(df)*100:.1f}%)   medie: ~{avg:.0f} cuvinte')

Filme incarcate: 40,197
Coloane: ['movie_id', 'title', 'overview', 'input_text', 'genres', 'keywords', 'director', 'release_date', 'runtime', 'popularity', 'tagline', 'review_texts', 'certification', 'cast', 'has_review', 'overview_summary', 'input_review', 'review_summary']

  overview               acoperire: 40,197 (100.0%)   medie: ~47 cuvinte
  overview_summary       acoperire: 40,197 (100.0%)   medie: ~26 cuvinte
  review_summary         acoperire: 11,644 (29.0%)   medie: ~11 cuvinte
  tagline                acoperire: 40,197 (100.0%)   medie: ~9 cuvinte
  keywords               acoperire: 40,197 (100.0%)   medie: ~10 cuvinte


In [4]:
def extract_cast_names(cast_raw, max_actors=4):
    if not isinstance(cast_raw, str) or not cast_raw.strip():
        return ''
    try:
        actors = ast.literal_eval(cast_raw)
        if isinstance(actors, list):
            names = [a['name'] for a in actors[:max_actors]
                     if isinstance(a, dict) and a.get('name')]
            return ', '.join(names)
    except Exception:
        pass
    return ''


def extract_keywords(kw_raw, max_kw=10):
    if not isinstance(kw_raw, str) or not kw_raw.strip():
        return ''
    try:
        parsed = ast.literal_eval(kw_raw)
        if isinstance(parsed, list):
            return ', '.join(str(k) for k in parsed[:max_kw])
    except Exception:
        pass
    return ', '.join(w.strip() for w in kw_raw.split(',')[:max_kw])


def parse_genres(genres_raw):
    if not isinstance(genres_raw, str) or not genres_raw.strip():
        return []
    try:
        parsed = ast.literal_eval(genres_raw)
        if isinstance(parsed, list):
            return [str(g).lower().strip() for g in parsed if g]
    except Exception:
        pass
    return [g.strip().lower() for g in genres_raw.split(',') if g.strip()]


def clean_overview_summary(title, ovs):
    t = str(title).strip()
    s = str(ovs).strip()
    if len(t) > 3 and s.lower().startswith(t.lower()):
        s = s[len(t):].lstrip('. \n').strip()
    return s if s else str(ovs).strip()


print('Functii ajutatoare definite.')

Functii ajutatoare definite.


In [5]:
def build_doc_text_v6(row):
    parts = [str(row.get('title', '')).strip() + '.']

    ovs = clean_overview_summary(row.get('title', ''), row.get('overview_summary', ''))
    ov  = str(row.get('overview', '')).replace('nan', '').strip()
    plot = ovs if ovs else ov
    if plot:
        parts.append('Plot: ' + plot)

    rev = str(row.get('review_summary', '')).replace('nan', '').strip()
    if rev:
        parts.append('Critics: ' + rev)

    genres = str(row.get('genres', '')).replace('nan', '').strip()
    if genres:
        parts.append('Genres: ' + genres)

    kw = extract_keywords(row.get('keywords', ''), 10)
    if kw:
        parts.append('Keywords: ' + kw)

    cast = extract_cast_names(row.get('cast', ''), 4)
    if cast:
        parts.append('Cast: ' + cast)

    return ' '.join(parts)


def build_doc_text_no_review(row):
    parts = [str(row.get('title', '')).strip() + '.']

    ovs = clean_overview_summary(row.get('title', ''), row.get('overview_summary', ''))
    ov  = str(row.get('overview', '')).replace('nan', '').strip()
    plot = ovs if ovs else ov
    if plot:
        parts.append('Plot: ' + plot)

    genres = str(row.get('genres', '')).replace('nan', '').strip()
    if genres:
        parts.append('Genres: ' + genres)

    kw = extract_keywords(row.get('keywords', ''), 10)
    if kw:
        parts.append('Keywords: ' + kw)

    cast = extract_cast_names(row.get('cast', ''), 4)
    if cast:
        parts.append('Cast: ' + cast)

    return ' '.join(parts)


df['doc_text']           = df.apply(build_doc_text_v6, axis=1)
df['doc_text_no_review'] = df.apply(build_doc_text_no_review, axis=1)

mask     = (df['tagline'] != '') & (df['doc_text'] != '')
df_valid = df[mask].copy().reset_index(drop=True)

print(f'Filme valide: {len(df_valid):,}')

real_leak = df_valid.apply(
    lambda r: len(str(r['tagline'])) > 15
              and str(r['tagline']).lower() in str(r['doc_text']).lower(),
    axis=1
).sum()
print(f'Leakage (tagline in doc_text): {real_leak} cazuri (~0.5% acceptabil)')

kw_cov = (df_valid['keywords'] != '').sum()
print(f'Filme cu keywords: {kw_cov:,} ({kw_cov/len(df_valid)*100:.1f}%)')

ex = df_valid[df_valid['keywords'] != ''].iloc[0]
print(f'\n--- Exemplu doc_text V6: {ex["title"]} ---')
print(ex['doc_text'])

Filme valide: 40,197
Leakage (tagline in doc_text): 127 cazuri (~0.5% acceptabil)
Filme cu keywords: 40,197 (100.0%)

--- Exemplu doc_text V6: Toy Story ---
Toy Story. Plot: Andys toys live happily in his room until Andys birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andys heart, Woody plots against Buzz. Critics: Tom Hanks leads a strong cast. Tim Allen is great, too, as Buzz Lightyear. Don Rickles (Mr. Potato Head), Wallace Shawn (Rex) and John Ratzenberger (Hamm) also bring fun. Genres: ['Family', 'Comedy', 'Animation', 'Adventure'] Keywords: rescue, friendship, mission, jealousy, villain, bullying, elementary school, rivalry, anthropomorphism, friends Cast: Tom Hanks, Tim Allen, Don Rickles, Jim Varney


In [6]:
from transformers import AutoTokenizer

tok    = AutoTokenizer.from_pretrained('BAAI/bge-base-en-v1.5')
sample = df_valid['doc_text'].sample(1000, random_state=42)
lengths = [len(tok(t, truncation=False)['input_ids']) for t in sample]

with_rev = df_valid[df_valid['review_summary'] != '']['doc_text'].sample(
    min(500, (df_valid['review_summary'] != '').sum()), random_state=42)
no_rev   = df_valid[df_valid['review_summary'] == '']['doc_text'].sample(
    min(500, (df_valid['review_summary'] == '').sum()), random_state=42)

len_with = [len(tok(t, truncation=False)['input_ids']) for t in with_rev]
len_no   = [len(tok(t, truncation=False)['input_ids']) for t in no_rev]

print(f'doc_text V6 CU review   — Medie: {np.mean(len_with):.0f} | Max: {max(len_with)} | >256: {sum(l>256 for l in len_with)}/{len(len_with)}')
print(f'doc_text V6 FARA review — Medie: {np.mean(len_no):.0f}   | Max: {max(len_no)}   | >256: {sum(l>256 for l in len_no)}/{len(len_no)}')
print(f'Global                  — Medie: {np.mean(lengths):.0f}  | 99p: {np.percentile(lengths,99):.0f}   | >256: {sum(l>256 for l in lengths)}/{len(lengths)}')
print(f'\nNota: max_length=256 => batch=128 e fezabil (ca V4b)')

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

doc_text V6 CU review   — Medie: 143 | Max: 189 | >256: 0/500
doc_text V6 FARA review — Medie: 80   | Max: 129   | >256: 0/500
Global                  — Medie: 98  | 99p: 171   | >256: 0/1000

Nota: max_length=256 => batch=128 e fezabil (ca V4b)


In [7]:
BGE_QUERY_PREFIX = 'Represent this sentence: '

train_df, eval_df = train_test_split(df_valid, test_size=0.1, random_state=42)
train_df = train_df.reset_index(drop=True)
eval_df  = eval_df.reset_index(drop=True)
print(f'Training: {len(train_df):,} filme')
print(f'Eval:     {len(eval_df):,} filme')

train_examples = [
    InputExample(texts=[BGE_QUERY_PREFIX + row['tagline'], row['doc_text']])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]

train_examples_no_review = [
    InputExample(texts=[BGE_QUERY_PREFIX + row['tagline'], row['doc_text_no_review']])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]

n_with = sum(1 for _, r in train_df.iterrows() if r['review_summary'] not in ('', 'nan'))
print(f'\nExemple principale:  {len(train_examples):,}')
print(f'  - cu review:       {n_with:,} ({n_with/len(train_df)*100:.1f}%)')
print(f'  - fara review:     {len(train_df)-n_with:,}')
print(f'\nExemplu pereche:')
ex = train_examples[0]
print(f'  Query: {ex.texts[0]}')
print(f'  Doc:   {ex.texts[1][:250]}...')

Training: 36,177 filme
Eval:     4,020 filme

Exemple principale:  35,788
  - cu review:       10,438 (28.9%)
  - fara review:     25,739

Exemplu pereche:
  Query: Represent this sentence: She loved and trusted him until he cut off her head.
  Doc:   Savage Intruder. Plot: An enigmatic young man manipulates his way into working at the decaying mansion of a once prolific, but now reclusive and alcoholic, movie star named Katharine Genres: ['Thriller', 'Horror'] Cast: Miriam Hopkins, David Garfield...


In [8]:
MODEL_NAME = 'BAAI/bge-base-en-v1.5'
OUTPUT_V6A = OUTPUT_PATH + 'sbert_v6a'
OUTPUT_V6B = OUTPUT_PATH + 'sbert_v6b'
device     = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

torch.cuda.empty_cache()
gc.collect()

sbert = SentenceTransformer(MODEL_NAME, device=str(device))

print(f'Model: {MODEL_NAME}')
print(f'Device: {device}')
print(f'Embedding dim: {sbert.encode(["test"]).shape[1]}')

total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
free_mem  = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3
print(f'GPU: {total_mem:.1f} GB total | {free_mem:.1f} GB libera')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model: BAAI/bge-base-en-v1.5
Device: cuda:0
Embedding dim: 768
GPU: 14.6 GB total | 14.1 GB libera


In [9]:
torch.cuda.empty_cache()
gc.collect()

sbert = sbert.to(device)
_tok  = sbert.tokenizer

def tokenize_batch(texts, max_length=256):
    return _tok(texts, padding=True, truncation=True,
                max_length=max_length, return_tensors='pt')

def get_embeddings(encoded):
    encoded = {k: v.to(device) for k, v in encoded.items()}
    out = sbert(encoded)
    return F.normalize(out['sentence_embedding'], p=2, dim=1)

def mnr_loss_fn(emb_a, emb_p, scale=50.0):
    scores = torch.mm(emb_a, emb_p.T) * scale
    labels = torch.arange(len(emb_a), device=device)
    return F.cross_entropy(scores, labels)

def simple_collate(examples):
    return [e.texts[0] for e in examples], [e.texts[1] for e in examples]

BATCH_SIZE = 128
EPOCHS     = 3

train_dl     = DataLoader(train_examples, batch_size=BATCH_SIZE,
                          shuffle=True, collate_fn=simple_collate)
total_steps  = len(train_dl) * EPOCHS
warmup_steps = int(0.1 * total_steps)

optimizer = AdamW(sbert.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler    = GradScaler('cuda')

print(f'Antrenare V6a: model={MODEL_NAME} | batch={BATCH_SIZE} | epoci={EPOCHS}')
print(f'  total steps={total_steps:,} | warmup={warmup_steps:,}')

for epoch in range(EPOCHS):
    sbert.train()
    total_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(train_dl), total=len(train_dl),
                desc=f'Epoch {epoch+1}/{EPOCHS}')
    for step, (anchors, positives) in pbar:
        enc_a = tokenize_batch(anchors)
        enc_p = tokenize_batch(positives)

        with autocast('cuda'):
            emb_a = get_embeddings(enc_a)
            emb_p = get_embeddings(enc_p)
            loss  = mnr_loss_fn(emb_a, emb_p)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(sbert.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    print(f'Epoch {epoch+1}/{EPOCHS} — Loss mediu: {total_loss/len(train_dl):.4f}')
    sbert.save(OUTPUT_PATH + f'sbert_v6a_epoch{epoch+1}')

sbert.save(OUTPUT_V6A)
print(f'\nV6a salvat: {OUTPUT_V6A}')

Antrenare V6a: model=BAAI/bge-base-en-v1.5 | batch=128 | epoci=3
  total steps=840 | warmup=84


Epoch 1/3:   0%|          | 0/280 [00:00<?, ?it/s]

Epoch 1/3 — Loss mediu: 2.6020


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 2/3:   0%|          | 0/280 [00:00<?, ?it/s]

...

Epoch 3/3:   0%|          | 0/280 [00:00<?, ?it/s]

Epoch 3/3 — Loss mediu: 1.9442


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


V6a salvat: /kaggle/working/sbert_v6a


In [10]:
BATCH_EMB = 256

queries_prefixed = [BGE_QUERY_PREFIX + t for t in df_valid['tagline'].tolist()]
docs             = df_valid['doc_text'].tolist()
docs_no_review   = df_valid['doc_text_no_review'].tolist()

sbert_base = SentenceTransformer(MODEL_NAME, device=str(device))
sbert_v6a  = SentenceTransformer(OUTPUT_V6A,  device=str(device))

print('Generare embeddings Baseline BGE...')
q_emb_base = sbert_base.encode(queries_prefixed, batch_size=BATCH_EMB,
                                show_progress_bar=True, convert_to_numpy=True,
                                normalize_embeddings=True)
d_emb_base = sbert_base.encode(docs, batch_size=BATCH_EMB,
                                show_progress_bar=True, convert_to_numpy=True,
                                normalize_embeddings=True)

print('\nGenerare embeddings V6a (fine-tuned, cu keywords)...')
q_emb_v6a = sbert_v6a.encode(queries_prefixed, batch_size=BATCH_EMB,
                               show_progress_bar=True, convert_to_numpy=True,
                               normalize_embeddings=True)
d_emb_v6a = sbert_v6a.encode(docs, batch_size=BATCH_EMB,
                               show_progress_bar=True, convert_to_numpy=True,
                               normalize_embeddings=True)

print('\nGenerare embeddings V6a ablatie (fara review)...')
d_emb_v6a_norev = sbert_v6a.encode(docs_no_review, batch_size=BATCH_EMB,
                                    show_progress_bar=True, convert_to_numpy=True,
                                    normalize_embeddings=True)

np.save(OUTPUT_PATH + 'q_emb_base.npy',      q_emb_base)
np.save(OUTPUT_PATH + 'd_emb_base.npy',      d_emb_base)
np.save(OUTPUT_PATH + 'q_emb_v6a.npy',       q_emb_v6a)
np.save(OUTPUT_PATH + 'd_emb_v6a.npy',       d_emb_v6a)
np.save(OUTPUT_PATH + 'd_emb_v6a_norev.npy', d_emb_v6a_norev)
print('\nEmbeddings salvate!')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generare embeddings Baseline BGE...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Generare embeddings V6a ablatie (fara review)...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Embeddings salvate!


In [11]:
df_valid['genres_list'] = df_valid['genres'].apply(parse_genres)

genre_to_indices = {}
for i, genres in enumerate(df_valid['genres_list']):
    for g in genres:
        genre_to_indices.setdefault(g, []).append(i)

print(f'Genuri unice: {len(genre_to_indices)}')


def evaluate_full_pool(query_emb, doc_emb, k_values=[1, 3, 5, 10, 20],
                       batch_size=512, label=''):
    N    = len(query_emb)
    hits = {k: 0 for k in k_values}
    max_k = max(k_values)

    for start in range(0, N, batch_size):
        end    = min(start + batch_size, N)
        scores = cosine_similarity(query_emb[start:end], doc_emb)
        top_k  = np.argpartition(scores, -max_k, axis=1)[:, -max_k:]

        for i, global_idx in enumerate(range(start, end)):
            top_sorted = top_k[i][np.argsort(scores[i][top_k[i]])[::-1]]
            for k in k_values:
                if global_idx in top_sorted[:k]:
                    hits[k] += 1

        if start % 10000 == 0 and start > 0:
            print(f'  {label} {start}/{N}...')

    return {k: hits[k] / N for k in k_values}


def evaluate_genre_filtered(query_emb, doc_emb, k_values=[1, 3, 5, 10, 20],
                             sample_n=3000, min_pool=30, label=''):
    np.random.seed(42)
    indices    = np.random.choice(len(query_emb), min(sample_n, len(query_emb)), replace=False)
    hits       = {k: 0 for k in k_values}
    pool_sizes = []

    for idx in indices:
        pool = set()
        for g in df_valid['genres_list'].iloc[idx]:
            pool.update(genre_to_indices.get(g, []))
        pool.add(idx)
        if len(pool) < min_pool:
            pool = set(range(len(df_valid)))

        pool_arr  = np.array(sorted(pool))
        pool_sizes.append(len(pool_arr))
        scores    = cosine_similarity(query_emb[idx:idx+1], doc_emb[pool_arr])[0]
        local_idx = int(np.where(pool_arr == idx)[0][0])
        ranked    = np.argsort(scores)[::-1]
        rank      = int(np.where(ranked == local_idx)[0][0]) + 1

        for k in k_values:
            if rank <= k:
                hits[k] += 1

    n = len(indices)
    print(f'  Pool mediu per query: {np.mean(pool_sizes):.0f} filme')
    return {k: hits[k] / n for k in k_values}


print('Functii de evaluare definite.')

Genuri unice: 19
Functii de evaluare definite.


In [12]:
results = {}

print('FULL POOL')

print('\nBaseline BGE (fara fine-tuning):')
results['base_full'] = evaluate_full_pool(q_emb_base, d_emb_base, label='base')
print(f'  {results["base_full"]}')

print('\nV6a (BGE fine-tuned, cu keywords):')
results['v6a_full'] = evaluate_full_pool(q_emb_v6a, d_emb_v6a, label='v6a')
print(f'  {results["v6a_full"]}')

print('\nV6a Ablatie (fara review):')
results['v6a_norev_full'] = evaluate_full_pool(q_emb_v6a, d_emb_v6a_norev, label='v6a-norev')
print(f'  {results["v6a_norev_full"]}')

print('\nGENRE-FILTERED POOL')

print('\nBaseline BGE:')
results['base_genre'] = evaluate_genre_filtered(q_emb_base, d_emb_base)
print(f'  {results["base_genre"]}')

print('\nV6a fine-tuned:')
results['v6a_genre'] = evaluate_genre_filtered(q_emb_v6a, d_emb_v6a)
print(f'  {results["v6a_genre"]}')

with open(OUTPUT_PATH + 'results_v6a.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nRezultate V6a salvate!')

FULL POOL

Baseline BGE (fara fine-tuning):
  {1: 0.0631390402268826, 3: 0.09468368286190512, 5: 0.11145110331616788, 10: 0.13717441600119412, 20: 0.16725128740950818}

V6a (BGE fine-tuned, cu keywords):
  {1: 0.10990869965420305, 3: 0.1685697937657039, 5: 0.2017563499763664, 10: 0.250615717590865, 20: 0.3098738711844167}

V6a Ablatie (fara review):
  {1: 0.11053063661467273, 3: 0.16710202253899545, 5: 0.19924372465606885, 10: 0.246635321043859, 20: 0.3030325646192502}

GENRE-FILTERED POOL

Baseline BGE:
  Pool mediu per query: 15835 filme
  {1: 0.081, 3: 0.12066666666666667, 5: 0.143, 10: 0.17466666666666666, 20: 0.206}

V6a fine-tuned:
  Pool mediu per query: 15835 filme
  {1: 0.12366666666666666, 3: 0.18, 5: 0.207, 10: 0.245, 20: 0.294}

Rezultate V6a salvate!


In [13]:
HN_PER_QUERY  = 6   
MINE_TOPK     = 60

train_taglines_prefixed = [
    BGE_QUERY_PREFIX + str(row['tagline'])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]
train_docs = [
    str(row['doc_text'])
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]
train_genres_list = [
    parse_genres(str(row['genres']))
    for _, row in train_df.iterrows()
    if len(str(row['tagline']).split()) >= 3
]

print('Encoding embeddings pentru mining...')
q_emb_train = sbert_v6a.encode(
    train_taglines_prefixed, batch_size=256,
    show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True
)
d_emb_train = sbert_v6a.encode(
    train_docs, batch_size=256,
    show_progress_bar=True, convert_to_numpy=True,
    normalize_embeddings=True
)

TARGET_SAME  = HN_PER_QUERY // 2   
TARGET_CROSS = HN_PER_QUERY - TARGET_SAME  

print(f'\nMining Mixed HN: {TARGET_SAME} same-genre + {TARGET_CROSS} cross-genre per query')
print(f'  Top-{MINE_TOPK} candidati bi-encoder | toate cele {len(train_taglines_prefixed):,} query-uri')

hard_neg_examples = []
skipped = 0

for i in tqdm(range(len(train_taglines_prefixed)), desc='Mining Mixed HN'):
    scores  = cosine_similarity(q_emb_train[i:i+1], d_emb_train)[0]
    scores[i] = -1.0
    top_idx = np.argsort(scores)[::-1][:MINE_TOPK]

    genres_q   = set(train_genres_list[i])
    same_negs  = []
    cross_negs = []

    for idx in top_idx:
        genres_c = set(train_genres_list[idx])
        common   = len(genres_q & genres_c)

        if common >= 1 and len(same_negs) < TARGET_SAME:
            same_negs.append(train_docs[idx])
        elif common == 0 and len(cross_negs) < TARGET_CROSS:
            cross_negs.append(train_docs[idx])

        if len(same_negs) >= TARGET_SAME and len(cross_negs) >= TARGET_CROSS:
            break

    for hn in same_negs + cross_negs:
        hard_neg_examples.append(
            InputExample(texts=[
                train_taglines_prefixed[i],
                train_docs[i],
                hn
            ])
        )

print(f'\nHard negative triplete generate: {len(hard_neg_examples):,}')
same_count  = sum(1 for i in range(0, len(hard_neg_examples), HN_PER_QUERY) for _ in range(TARGET_SAME))
print(f'  Estimat ~50% same-genre, ~50% cross-genre')
if hard_neg_examples:
    ex = hard_neg_examples[0]
    print(f'\nExemplu triplet:')
    print(f'  Anchor:  {ex.texts[0]}')
    print(f'  Pozitiv: {ex.texts[1][:120]}...')
    print(f'  HardNeg: {ex.texts[2][:120]}...')

Encoding embeddings pentru mining...


Batches:   0%|          | 0/140 [00:00<?, ?it/s]

Batches:   0%|          | 0/140 [00:00<?, ?it/s]


Mining Mixed HN: 3 same-genre + 3 cross-genre per query
  Top-60 candidati bi-encoder | toate cele 35,788 query-uri


Mining Mixed HN:   0%|          | 0/35788 [00:00<?, ?it/s]


Hard negative triplete generate: 200,243
  Estimat ~50% same-genre, ~50% cross-genre

Exemplu triplet:
  Anchor:  Represent this sentence: She loved and trusted him until he cut off her head.
  Pozitiv: Savage Intruder. Plot: An enigmatic young man manipulates his way into working at the decaying mansion of a once prolifi...
  HardNeg: Victim of Love. Plot: Therapist Tess falls for widowed professor Paul, whos having an affair with Carla, one of her pati...


In [14]:
for var in ['sbert', 'sbert_base', 'sbert_v6a']:
    if var in globals():
        del globals()[var]
torch.cuda.empty_cache()
gc.collect()

free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3
print(f'GPU libera dupa curatare: {free:.2f} GB')

sbert_v6b = SentenceTransformer(OUTPUT_V6A, device=str(device))
sbert_v6b = sbert_v6b.to(device)
_tok_v6b  = sbert_v6b.tokenizer

def tokenize_v6b(texts, max_length=256):
    return _tok_v6b(texts, padding=True, truncation=True,
                    max_length=max_length, return_tensors='pt')

def get_emb_v6b(encoded):
    encoded = {k: v.to(device) for k, v in encoded.items()}
    out = sbert_v6b(encoded)
    return F.normalize(out['sentence_embedding'], p=2, dim=1)

def mnr_triplet_loss(emb_a, emb_p, emb_n, scale=50.0):
    emb_docs = torch.cat([emb_p, emb_n], dim=0)
    scores   = torch.mm(emb_a, emb_docs.T) * scale
    labels   = torch.arange(len(emb_a), device=device)
    return F.cross_entropy(scores, labels)

def hn_collate(examples):
    return ([e.texts[0] for e in examples],
            [e.texts[1] for e in examples],
            [e.texts[2] for e in examples])

BATCH_HN  = 64
EPOCHS_HN = 2

hn_dl           = DataLoader(hard_neg_examples, batch_size=BATCH_HN,
                              shuffle=True, collate_fn=hn_collate)
total_steps_hn  = len(hn_dl) * EPOCHS_HN
warmup_steps_hn = int(0.05 * total_steps_hn)

optimizer_hn = AdamW(sbert_v6b.parameters(), lr=1e-5, weight_decay=0.01)
scheduler_hn = get_linear_schedule_with_warmup(optimizer_hn, warmup_steps_hn, total_steps_hn)
scaler_hn    = GradScaler('cuda')

print(f'Fine-tuning V6b cu Mixed Hard Negatives:')
print(f'  Start din: {OUTPUT_V6A}')
print(f'  batch={BATCH_HN} | triplete={len(hard_neg_examples):,} | epoci={EPOCHS_HN}')

for epoch in range(EPOCHS_HN):
    sbert_v6b.train()
    total_loss = 0.0
    optimizer_hn.zero_grad()

    pbar = tqdm(enumerate(hn_dl), total=len(hn_dl),
                desc=f'V6b Epoch {epoch+1}/{EPOCHS_HN}')
    for step, (anchors, positives, negatives) in pbar:
        enc_a = tokenize_v6b(anchors)
        enc_p = tokenize_v6b(positives)
        enc_n = tokenize_v6b(negatives)

        with autocast('cuda'):
            emb_a = get_emb_v6b(enc_a)
            emb_p = get_emb_v6b(enc_p)
            emb_n = get_emb_v6b(enc_n)
            loss  = mnr_triplet_loss(emb_a, emb_p, emb_n)

        scaler_hn.scale(loss).backward()
        scaler_hn.unscale_(optimizer_hn)
        torch.nn.utils.clip_grad_norm_(sbert_v6b.parameters(), max_norm=1.0)
        scaler_hn.step(optimizer_hn)
        scaler_hn.update()
        scheduler_hn.step()
        optimizer_hn.zero_grad()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    print(f'V6b Epoch {epoch+1} — Loss mediu: {total_loss/len(hn_dl):.4f}')

sbert_v6b.save(OUTPUT_V6B)
print(f'\nV6b salvat: {OUTPUT_V6B}')

GPU libera dupa curatare: 13.33 GB


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fine-tuning V6b cu Mixed Hard Negatives:
  Start din: /kaggle/working/sbert_v6a
  batch=64 | triplete=200,243 | epoci=2


V6b Epoch 1/2:   0%|          | 0/3129 [00:00<?, ?it/s]

V6b Epoch 1 — Loss mediu: 2.1993


V6b Epoch 2/2:   0%|          | 0/3129 [00:00<?, ?it/s]

V6b Epoch 2 — Loss mediu: 1.5626


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


V6b salvat: /kaggle/working/sbert_v6b


In [15]:
sbert_v6b_eval = SentenceTransformer(OUTPUT_V6B, device=str(device))

print('Generare embeddings V6b...')
q_emb_v6b = sbert_v6b_eval.encode(queries_prefixed, batch_size=BATCH_EMB,
                                    show_progress_bar=True, convert_to_numpy=True,
                                    normalize_embeddings=True)
d_emb_v6b = sbert_v6b_eval.encode(docs, batch_size=BATCH_EMB,
                                    show_progress_bar=True, convert_to_numpy=True,
                                    normalize_embeddings=True)

np.save(OUTPUT_PATH + 'q_emb_v6b.npy', q_emb_v6b)
np.save(OUTPUT_PATH + 'd_emb_v6b.npy', d_emb_v6b)

print('\nEvaluare V6b full pool...')
results['v6b_full'] = evaluate_full_pool(q_emb_v6b, d_emb_v6b, label='v6b')
print(f'  {results["v6b_full"]}')

print('\nEvaluare V6b genre-filtered...')
results['v6b_genre'] = evaluate_genre_filtered(q_emb_v6b, d_emb_v6b)
print(f'  {results["v6b_genre"]}')

with open(OUTPUT_PATH + 'results_v6b.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nRezultate V6b salvate!')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generare embeddings V6b...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Evaluare V6b full pool...
  {1: 0.1932233748787223, 3: 0.30375401149339504, 5: 0.3602258875040426, 10: 0.4427942383759982, 20: 0.5288454362265841}

Evaluare V6b genre-filtered...
  Pool mediu per query: 15835 filme
  {1: 0.102, 3: 0.14433333333333334, 5: 0.16966666666666666, 10: 0.212, 20: 0.249}

Rezultate V6b salvate!


In [16]:
SAMPLE_BS  = 300
BATCH_BERT = 64

np.random.seed(42)
sample_idx = np.random.choice(len(df_valid), SAMPLE_BS, replace=False)

def get_top1_idx(q_emb, d_emb, idx):
    scores = cosine_similarity(q_emb[idx:idx+1], d_emb)[0]
    return int(np.argmax(scores))

def get_bert_text(idx):
    row = df_valid.iloc[idx]
    return clean_overview_summary(row['title'], row['overview_summary'])

refs      = [get_bert_text(i) for i in sample_idx]
hyps_base = [get_bert_text(get_top1_idx(q_emb_base, d_emb_base, i)) for i in sample_idx]
hyps_v6a  = [get_bert_text(get_top1_idx(q_emb_v6a,  d_emb_v6a,  i)) for i in sample_idx]
hyps_v6b  = [get_bert_text(get_top1_idx(q_emb_v6b,  d_emb_v6b,  i)) for i in sample_idx]

hit1_base = [1 if get_top1_idx(q_emb_base, d_emb_base, i) == i else 0 for i in sample_idx]
hit1_v6a  = [1 if get_top1_idx(q_emb_v6a,  d_emb_v6a,  i) == i else 0 for i in sample_idx]
hit1_v6b  = [1 if get_top1_idx(q_emb_v6b,  d_emb_v6b,  i) == i else 0 for i in sample_idx]

print(f'Sample BERTScore: {SAMPLE_BS} filme')
print(f'Hit@1 — base: {sum(hit1_base)/len(hit1_base):.3f} | v6a: {sum(hit1_v6a)/len(hit1_v6a):.3f} | v6b: {sum(hit1_v6b)/len(hit1_v6b):.3f}')

bert_results = {}
for name, hyps in [('base', hyps_base), ('v6a', hyps_v6a), ('v6b', hyps_v6b)]:
    print(f'\nBERTScore {name}...')
    _, _, F1 = bert_score_compute(
        hyps, refs, lang='en',
        model_type='bert-base-uncased',
        batch_size=BATCH_BERT, verbose=False
    )
    bert_results[name] = F1.numpy()
    hits_list = {'base': hit1_base, 'v6a': hit1_v6a, 'v6b': hit1_v6b}[name]
    f1_miss   = F1.numpy()[[h == 0 for h in hits_list]]
    print(f'  F1 global: {F1.mean():.4f} +- {F1.std():.4f}')
    if len(f1_miss) > 0:
        print(f'  F1 miss:   {f1_miss.mean():.4f}')

print('\nBERTScore calculat!')

Sample BERTScore: 300 filme
Hit@1 — base: 0.077 | v6a: 0.107 | v6b: 0.107

BERTScore base...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F1 global: 0.5075 +- 0.1772
  F1 miss:   0.4487

BERTScore v6b...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  F1 global: 0.5038 +- 0.1784
  F1 miss:   0.4445

BERTScore calculat!


In [17]:
print(f'{"COMPARATIE MODELE — V4b vs V6":^80}')
print(f'{"Model":<42} {"Pool":<7} {"Hit@1":>6} {"Hit@5":>6} {"Hit@10":>7} {"Hit@20":>7}')

ref_vals = {
    '[Ref] V2 all-mpnet MNR':       {1: 0.100, 5: 0.187, 10: 0.236, 20: 0.293},
    '[Ref] V3 all-mpnet + review':  {1: None,  5: None,  10: 0.237, 20: None},
    '[Ref] V4a BGE (batch=128)':    {1: 0.104, 5: 0.191, 10: 0.238, 20: 0.294},
    '[Ref] V4b BGE + HN cross':     {1: 0.164, 5: 0.313, 10: 0.387, 20: 0.467},
    '[Ref] V4b genre-filtered':     {1: 0.100, 5: 0.166, 10: 0.198, 20: 0.240},
}

rows_hk = [
    ('Baseline BGE (no fine-tune)',         'full',  'base_full'),
    ('[Ref] V2 all-mpnet MNR',              'full',  None),
    ('[Ref] V3 all-mpnet + review',         'full',  None),
    ('[Ref] V4a BGE (batch=128)',           'full',  None),
    ('[Ref] V4b BGE + HN cross',           'full',  None),
    ('V6a BGE + Keywords (batch=128)',     'full',  'v6a_full'),
    ('V6a Ablatie (fara review)',          'full',  'v6a_norev_full'),
    ('V6b BGE + Keywords + Mixed HN',     'full',  'v6b_full'),
    ('Baseline BGE',                       'genre', 'base_genre'),
    ('[Ref] V4b genre-filtered',          'genre', None),
    ('V6a fine-tuned',                    'genre', 'v6a_genre'),
    ('V6b + Mixed HN',                   'genre', 'v6b_genre'),
]

for label, pool, key in rows_hk:
    if key and key in results:
        r   = results[key]
        h1  = f"{r[1]*100:.1f}%"
        h5  = f"{r[5]*100:.1f}%"
        h10 = f"{r[10]*100:.1f}%"
        h20 = f"{r[20]*100:.1f}%"
    elif label in ref_vals:
        rv  = ref_vals[label]
        h1  = f"{rv[1]*100:.1f}%" if rv[1]  else '    —'
        h5  = f"{rv[5]*100:.1f}%" if rv[5]  else '    —'
        h10 = f"{rv[10]*100:.1f}%" if rv[10] else '    —'
        h20 = f"{rv[20]*100:.1f}%" if rv[20] else '    —'
    else:
        continue
    print(f'{label:<42} {pool:<7} {h1:>6} {h5:>6} {h10:>7} {h20:>7}')

print()
print(f'{"BERTSCORE F1 (overview_summary, N=300)":^65}')
print(f'{"Model":<30} {"BS-F1 global":>13} {"BS-F1 miss":>12}')
for name, hits_list, label in [
    ('base', hit1_base, 'Baseline BGE'),
    ('v6a',  hit1_v6a,  'V6a + Keywords'),
    ('v6b',  hit1_v6b,  'V6b + Keywords + Mixed HN'),
]:
    f1_all  = bert_results[name]
    f1_miss = bert_results[name][[h == 0 for h in hits_list]]
    print(f'{label:<30} {f1_all.mean():>13.4f} {f1_miss.mean():>12.4f}')

                         COMPARATIE MODELE — V4b vs V6                          
Model                                      Pool     Hit@1  Hit@5  Hit@10  Hit@20
Baseline BGE (no fine-tune)                full      6.3%  11.1%   13.7%   16.7%
[Ref] V2 all-mpnet MNR                     full     10.0%  18.7%   23.6%   29.3%
[Ref] V3 all-mpnet + review                full         —      —   23.7%       —
[Ref] V4a BGE (batch=128)                  full     10.4%  19.1%   23.8%   29.4%
[Ref] V4b BGE + HN cross                   full     16.4%  31.3%   38.7%   46.7%
V6a BGE + Keywords (batch=128)             full     11.0%  20.2%   25.1%   31.0%
V6a Ablatie (fara review)                  full     11.1%  19.9%   24.7%   30.3%
V6b BGE + Keywords + Mixed HN              full     19.3%  36.0%   44.3%   52.9%
Baseline BGE                               genre     8.1%  14.3%   17.5%   20.6%
[Ref] V4b genre-filtered                   genre    10.0%  16.6%   19.8%   24.0%
V6a fine-tuned              

In [18]:
def recommend_v6(query: str, top_k: int = 5):
    q_prefixed = BGE_QUERY_PREFIX + query
    q_emb      = sbert_v6b_eval.encode([q_prefixed], convert_to_numpy=True,
                                        normalize_embeddings=True)
    bi_scores  = cosine_similarity(q_emb, d_emb_v6b)[0]
    top_idx    = np.argsort(bi_scores)[::-1][:top_k]

    print(f'\nQuery: "{query}"')
    print(f'{"Rang":<5} {"Rev":>4} {"Titlu":<40} {"Gen":<28} {"Scor":>6}')

    for rank, idx in enumerate(top_idx, 1):
        row     = df_valid.iloc[idx]
        rev_ico = 'Yes' if row['review_summary'] not in ('', 'nan') else '  '
        genres  = str(row['genres'])[:26]
        score   = bi_scores[idx]
        print(f'{rank:<5} {rev_ico:>4} {str(row["title"]):<40} {genres:<28} {score:>6.3f}')


demo_queries = [
    'animated toys that come to life and go on adventures',
    'a superhero saves the world from an alien invasion',
    'romantic comedy set in New York with funny misunderstandings',
    'psychological thriller about dreams and reality',
    'a young wizard discovers his magical powers at school',
]

for q in demo_queries:
    recommend_v6(q, top_k=5)


Query: "animated toys that come to life and go on adventures"
Rang   Rev Titlu                                    Gen                            Scor
1          Barbie Presents: Thumbelina              ['Animation', 'Family']       0.722
2          Toopy and Binoo The Movie                ['Animation', 'Adventure',    0.715
3          The Indian in the Cupboard               ['Adventure', 'Family', 'F    0.709
4          The Hairy Tooth Fairy                    ['Family', 'Animation']       0.703
5          Barbie & Her Sisters in A Pony Tale      ['Animation', 'Family']       0.698

Query: "a superhero saves the world from an alien invasion"
Rang   Rev Titlu                                    Gen                            Scor
1      Yes The Guyver                               ['Science Fiction', 'Actio    0.694
2      Yes Howard the Duck                          ['Fantasy', 'Comedy', 'Act    0.690
3          Star Kid                                 ['Action', 'Adventure', 'F    0.

In [2]:
import numpy as np
import pandas as pd
import ast
from sklearn.model_selection import train_test_split

df = pd.read_csv('/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv')
for col in ['overview', 'overview_summary', 'review_summary', 'tagline', 'genres', 'keywords', 'cast']:
    df[col] = df[col].fillna('').astype(str).str.strip().replace('nan', '')

def clean_overview_summary(title, ovs):
    t = str(title).strip()
    s = str(ovs).strip()
    if len(t) > 3 and s.lower().startswith(t.lower()):
        s = s[len(t):].lstrip('. \n').strip()
    return s if s else str(ovs).strip()

def extract_cast_names(cast_raw, max_actors=4):
    if not isinstance(cast_raw, str) or not cast_raw.strip():
        return ''
    try:
        actors = ast.literal_eval(cast_raw)
        if isinstance(actors, list):
            return ', '.join(a['name'] for a in actors[:max_actors] if isinstance(a, dict) and a.get('name'))
    except:
        pass
    return ''

def extract_keywords(kw_raw, max_kw=10):
    if not isinstance(kw_raw, str) or not kw_raw.strip():
        return ''
    try:
        parsed = ast.literal_eval(kw_raw)
        if isinstance(parsed, list):
            return ', '.join(str(k) for k in parsed[:max_kw])
    except:
        pass
    return ''

def build_doc_text_v6(row):
    parts = [str(row.get('title', '')).strip() + '.']
    ovs = clean_overview_summary(row.get('title', ''), row.get('overview_summary', ''))
    ov  = str(row.get('overview', '')).replace('nan', '').strip()
    plot = ovs if ovs else ov
    if plot: parts.append('Plot: ' + plot)
    rev = str(row.get('review_summary', '')).replace('nan', '').strip()
    if rev: parts.append('Critics: ' + rev)
    genres = str(row.get('genres', '')).replace('nan', '').strip()
    if genres: parts.append('Genres: ' + genres)
    kw = extract_keywords(row.get('keywords', ''), 10)
    if kw: parts.append('Keywords: ' + kw)
    cast = extract_cast_names(row.get('cast', ''), 4)
    if cast: parts.append('Cast: ' + cast)
    return ' '.join(parts)

df['doc_text'] = df.apply(build_doc_text_v6, axis=1)
df_valid = df[(df['tagline'] != '') & (df['doc_text'] != '')].copy().reset_index(drop=True)

_, eval_df = train_test_split(df_valid, test_size=0.1, random_state=42)
eval_df = eval_df.reset_index(drop=True)

full_id_to_idx = {mid: i for i, mid in enumerate(df_valid['movie_id'].tolist())}
eval_indices = [full_id_to_idx[mid] for mid in eval_df['movie_id'].tolist()]

def eval_model(q_emb_path, d_emb_path, label, comparatie):
    q_emb = np.load(q_emb_path)
    d_emb = np.load(d_emb_path)
    test_q_emb = q_emb[eval_indices]
    hits = {k: 0 for k in [1, 5, 10, 20]}
    n = len(eval_df)
    for i, gi in enumerate(eval_indices):
        sims = test_q_emb[i] @ d_emb.T
        rank = int((sims > sims[gi]).sum()) + 1
        for k in hits:
            if rank <= k:
                hits[k] += 1
    print(f'\n{label} pe filme nevazute vs pool complet 40k:')
    for k in [1, 5, 10, 20]:
        print(f'  Hit@{k} = {hits[k]/n:.1%}')
    print(f'Comparatie: {label} pe toti 40k = {comparatie}')

BASE = '/kaggle/input/notebooks/catalinalupu/notebook-varianta6/'

eval_model(BASE + 'q_emb_v6a.npy', BASE + 'd_emb_v6a.npy', 'V6a (MNR, cu keywords)', '25.1%')
eval_model(BASE + 'q_emb_v6b.npy', BASE + 'd_emb_v6b.npy', 'V6b (MNR + Mixed HN)', '44.3%')


V6a (MNR, cu keywords) pe filme nevazute vs pool complet 40k:
  Hit@1 = 10.3%
  Hit@5 = 17.9%
  Hit@10 = 21.8%
  Hit@20 = 26.5%
Comparatie: V6a (MNR, cu keywords) pe toti 40k = 25.1%

V6b (MNR + Mixed HN) pe filme nevazute vs pool complet 40k:
  Hit@1 = 9.2%
  Hit@5 = 15.1%
  Hit@10 = 18.8%
  Hit@20 = 22.6%
Comparatie: V6b (MNR + Mixed HN) pe toti 40k = 44.3%


In [5]:
!pip install bert-score --quiet
import numpy as np
import pandas as pd
import torch
import ast
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from bert_score import score as bert_score_compute

INPUT_NB   = '/kaggle/input/notebooks/catalinalupu/notebook-varianta6/'
INPUT_CSV  = '/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv'
OUTPUT_PATH = '/kaggle/working/'
BATCH_EMB  = 256
BATCH_BERT = 64
SAMPLE_BS  = 300
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

def extract_keywords(kw_raw, max_kw=10):
    if not isinstance(kw_raw, str) or not kw_raw.strip(): return ''
    try:
        parsed = ast.literal_eval(kw_raw)
        if isinstance(parsed, list): return ', '.join(str(k) for k in parsed[:max_kw])
    except: pass
    return ', '.join(w.strip() for w in kw_raw.split(',')[:max_kw])

def extract_cast_names(cast_raw, max_actors=4):
    if not isinstance(cast_raw, str) or not cast_raw.strip(): return ''
    try:
        actors = ast.literal_eval(cast_raw)
        if isinstance(actors, list):
            return ', '.join(a['name'] for a in actors[:max_actors]
                             if isinstance(a, dict) and a.get('name'))
    except: pass
    return ''

def clean_overview_summary(title, ovs):
    t, s = str(title).strip(), str(ovs).strip()
    if len(t) > 3 and s.lower().startswith(t.lower()):
        s = s[len(t):].lstrip('. \n').strip()
    return s if s else str(ovs).strip()

def build_doc_text_no_review(row):
    parts = [str(row.get('title', '')).strip() + '.']
    ovs  = clean_overview_summary(row.get('title',''), row.get('overview_summary',''))
    ov   = str(row.get('overview','')).replace('nan','').strip()
    plot = ovs if ovs else ov
    if plot:   parts.append('Plot: ' + plot)
    genres = str(row.get('genres','')).replace('nan','').strip()
    if genres: parts.append('Genres: ' + genres)
    kw = extract_keywords(row.get('keywords',''), 10)
    if kw:     parts.append('Keywords: ' + kw)
    cast = extract_cast_names(row.get('cast',''), 4)
    if cast:   parts.append('Cast: ' + cast)
    return ' '.join(parts)

def get_top1_idx(q_emb, d_emb, idx):
    scores = cosine_similarity(q_emb[idx:idx+1], d_emb)[0]
    return int(np.argmax(scores))

def get_bert_text(idx):
    row = df_valid.iloc[idx]
    return clean_overview_summary(row['title'], row['overview_summary'])

print('Încărcare CSV...')
df = pd.read_csv(INPUT_CSV)
for col in ['overview','tagline','review_summary','overview_summary','genres','keywords','cast']:
    df[col] = df[col].fillna('').astype(str).str.strip().replace('nan','')
df['doc_text_no_review'] = df.apply(build_doc_text_no_review, axis=1)
mask = (df['tagline'] != '') & (df['doc_text_no_review'] != '')
df_valid = df[mask].copy().reset_index(drop=True)
docs_no_review = df_valid['doc_text_no_review'].tolist()
print(f'df_valid: {len(df_valid):,} filme')

print('Încărcare embeddings...')
q_emb_v6b = np.load(INPUT_NB + 'q_emb_v6b.npy')
d_emb_v6b = np.load(INPUT_NB + 'd_emb_v6b.npy')
print(f'q_emb_v6b: {q_emb_v6b.shape}')

print('Încărcare model V6b...')
sbert_v6b_eval = SentenceTransformer(INPUT_NB + 'sbert_v6b', device=str(device))

print('Generare d_emb_v6b_norev...')
d_emb_v6b_norev = sbert_v6b_eval.encode(
    docs_no_review, batch_size=BATCH_EMB,
    show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True
)
np.save(OUTPUT_PATH + 'd_emb_v6b_norev.npy', d_emb_v6b_norev)
print('Salvat: d_emb_v6b_norev.npy')

np.random.seed(42)
sample_idx = np.random.choice(len(df_valid), SAMPLE_BS, replace=False)
refs           = [get_bert_text(i) for i in sample_idx]
hyps_v6b_norev = [get_bert_text(get_top1_idx(q_emb_v6b, d_emb_v6b_norev, i)) for i in sample_idx]
hit1_v6b_norev = [1 if get_top1_idx(q_emb_v6b, d_emb_v6b_norev, i) == i else 0 for i in sample_idx]
print(f'Hit@1 pe sample (v6b_norev): {sum(hit1_v6b_norev)/len(hit1_v6b_norev):.3f}')

print('Calculare BERTScore v6b_norev...')
_, _, F1_norev = bert_score_compute(
    hyps_v6b_norev, refs, lang='en',
    model_type='bert-base-uncased',
    batch_size=BATCH_BERT, verbose=False
)
f1_all  = F1_norev.numpy()
f1_miss = f1_all[[h == 0 for h in hit1_v6b_norev]]
print(f'\n  F1 global: {f1_all.mean():.4f} +- {f1_all.std():.4f}')
print(f'  F1 miss:   {f1_miss.mean():.4f}')

print(f'{"Model":<30} {"BS-F1 global":>13}')
print(f'{"Baseline BGE":<30} {"0.4708":>13}')
print(f'{"V6b (cu review)":<30} {"0.5038":>13}')
print(f'{"V6b_norev (fara review)":<30} {f1_all.mean():>13.4f}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 76.2 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cud

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generare d_emb_v6b_norev...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Salvat: d_emb_v6b_norev.npy


...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  F1 global: 0.5036 +- 0.1803
  F1 miss:   0.4422
Model                           BS-F1 global
Baseline BGE                          0.4708
V6b (cu review)                       0.5038
V6b_norev (fara review)               0.5036


In [1]:
!pip install bert-score --quiet

import numpy as np
import pandas as pd
import torch
import ast
from sklearn.metrics.pairwise import cosine_similarity
from bert_score import score as bert_score_compute

INPUT_NB  = '/kaggle/input/notebooks/catalinalupu/notebook-varianta6/'
INPUT_CSV = '/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv'
BATCH_BERT = 64
SAMPLE_BS  = 300

def clean_overview_summary(title, ovs):
    t, s = str(title).strip(), str(ovs).strip()
    if len(t) > 3 and s.lower().startswith(t.lower()):
        s = s[len(t):].lstrip('. \n').strip()
    return s if s else str(ovs).strip()

def get_top1_idx(q_emb, d_emb, idx):
    scores = cosine_similarity(q_emb[idx:idx+1], d_emb)[0]
    return int(np.argmax(scores))

def get_bert_text(idx):
    row = df_valid.iloc[idx]
    return clean_overview_summary(row['title'], row['overview_summary'])

print('Încărcare CSV...')
df = pd.read_csv(INPUT_CSV)
for col in ['overview','tagline','overview_summary']:
    df[col] = df[col].fillna('').astype(str).str.strip().replace('nan','')
mask = df['tagline'] != ''
df_valid = df[mask].copy().reset_index(drop=True)
print(f'df_valid: {len(df_valid):,} filme')

print('Încărcare embeddings...')
q_emb_v6a      = np.load(INPUT_NB + 'q_emb_v6a.npy')
d_emb_v6a      = np.load(INPUT_NB + 'd_emb_v6a.npy')
d_emb_v6a_norev = np.load(INPUT_NB + 'd_emb_v6a_norev.npy')
print(f'Shape: {q_emb_v6a.shape}')

np.random.seed(42)
sample_idx = np.random.choice(len(df_valid), SAMPLE_BS, replace=False)
refs = [get_bert_text(i) for i in sample_idx]

hyps_v6a       = [get_bert_text(get_top1_idx(q_emb_v6a, d_emb_v6a,       i)) for i in sample_idx]
hyps_v6a_norev = [get_bert_text(get_top1_idx(q_emb_v6a, d_emb_v6a_norev, i)) for i in sample_idx]
hit1_v6a       = [1 if get_top1_idx(q_emb_v6a, d_emb_v6a,       i) == i else 0 for i in sample_idx]
hit1_v6a_norev = [1 if get_top1_idx(q_emb_v6a, d_emb_v6a_norev, i) == i else 0 for i in sample_idx]

print('BERTScore V6a...')
_, _, F1_v6a = bert_score_compute(hyps_v6a, refs, lang='en',
    model_type='bert-base-uncased', batch_size=BATCH_BERT, verbose=False)

print('BERTScore V6a_norev...')
_, _, F1_norev = bert_score_compute(hyps_v6a_norev, refs, lang='en',
    model_type='bert-base-uncased', batch_size=BATCH_BERT, verbose=False)

print(f'{"Model":<30} {"BS-F1 global":>13}')
print(f'{"V6a (cu review)":<30} {F1_v6a.mean():>13.4f}')
print(f'{"V6a_norev (fara review)":<30} {F1_norev.mean():>13.4f}')
print(f'{"Diferenta":<30} {(F1_v6a.mean()-F1_norev.mean()):>+13.4f}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 106.4 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cu

BERTScore V6a...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERTScore V6a_norev...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model                           BS-F1 global
V6a (cu review)                       0.5075
V6a_norev (fara review)               0.5090
Diferenta                            -0.0016
